# Tract-Based Spatial Statistics (TBSS)

TBSS is the most widely used method for comparing DTI metrics (FA, MD, RD, AD) across groups. If you read dMRI papers, you have seen TBSS results — the coloured blobs on a white matter skeleton image.

## The core idea

Comparing FA voxel-by-voxel across subjects is problematic because:
1. Subjects have slightly different brain shapes — registration is never perfect
2. Small misalignments create false positives at tract boundaries
3. The full FA image has ~500,000 voxels — massive multiple comparison problem

TBSS solves this by **projecting all subjects' FA onto a common white matter skeleton** — a thin 1-voxel-wide representation of the centre of each white matter tract. This:
- Reduces the number of comparisons (only skeleton voxels)
- Makes the result robust to small registration errors
- Gives results that are anatomically interpretable

## The full TBSS pipeline (FSL)

```
Per subject:                         Group:
  raw DWI                            all FA images
    ↓ (BET + eddy + dtifit)              ↓ tbss_1_preproc
  FA map                             eroded + slicesdir QC
                                         ↓ tbss_2_reg
                                     registered to MNI152
                                         ↓ tbss_3_postreg
                                     mean FA + skeleton
                                         ↓ tbss_4_prestats
                                     skeleton projections
                                         ↓ randomise
                                     voxelwise stats (TFCE)
```

> **FSL is the only tool with native TBSS support.** MRtrix3 and DIPY do not implement TBSS — this is a clear FSL strength. MRtrix3's answer is Fixel-Based Analysis (next notebook).

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

# In a real study: one FA image per subject
# Here we simulate 10 subjects from our phantom
dti_dir   = Path('../../data/hcp/100307/dti')
tbss_dir  = Path('../../data/group/tbss')
tbss_dir.mkdir(parents=True, exist_ok=True)

print('TBSS requires one FA.nii.gz per subject in a single directory.')
print('Naming convention: <subjectID>_FA.nii.gz')

## Step 0: Prepare per-subject FA images

Before TBSS, every subject needs:
1. BET brain extraction ← Module 1
2. Eddy correction ← Module 1  
3. DTI fitting → FA map ← Module 2

For a real study with N subjects, you would loop:

In [ ]:
# ─── Simulate multiple subjects for demonstration ─────────────────────────────
# In a real study: replace with your actual subject list

import shutil

fa_source = dti_dir / 'fsl_dti_FA.nii.gz'

if fa_source.exists():
    # Copy and add synthetic noise to simulate 10 subjects
    img  = nib.load(str(fa_source))
    fa   = img.get_fdata()
    rng  = np.random.default_rng(42)

    subjects = [f'sub-{i:02d}' for i in range(1, 11)]
    for i, sub in enumerate(subjects):
        # Simulate inter-subject variability
        noise = rng.normal(0, 0.03, fa.shape).astype(np.float32)
        fa_sub = np.clip(fa + noise, 0, 1)
        out    = tbss_dir / f'{sub}_FA.nii.gz'
        nib.save(nib.Nifti1Image(fa_sub, img.affine), str(out))

    print(f'Created {len(subjects)} simulated FA images in {tbss_dir}')
    print('Files:', [f.name for f in sorted(tbss_dir.glob('*_FA.nii.gz'))])
else:
    print('Run Module 2 (DTI) first to generate FA maps.')
    print()
    # Real-study loop template:
    print('=== Real study template ===')
    template = '''
for sub in subjects:
    # 1. Brain extraction
    subprocess.run(['bet', f'{sub}/b0.nii.gz', f'{sub}/b0_brain', '-f', '0.25', '-m', '-R'])
    # 2. Eddy correction
    subprocess.run(['eddy_openmp', '--imain=...', '--out=...', '--repol'])
    # 3. DTI fit → FA
    subprocess.run(['dtifit', '--data=...', '--out=...', '--wls'])
    # 4. Copy FA to TBSS directory
    shutil.copy(f'{sub}/dtifit_FA.nii.gz', f'tbss/{sub}_FA.nii.gz')
'''
    print(template)

## Step 1: tbss_1_preproc — erode and check

Erodes the edges of all FA maps (removes unreliable boundary voxels) and creates a QC slicesdir.

In [ ]:
# ─── [FSL] tbss_1_preproc ─────────────────────────────────────────────────────
#
# Must be run FROM the tbss directory (it works on all *.nii.gz in cwd)
# Creates: FA/ subdirectory with preprocessed images
#          slicesdir/ with QC images

step1_cmd = ['tbss_1_preproc', '*.nii.gz']

print('[FSL] Step 1: tbss_1_preproc')
print(f'  Run from: {tbss_dir}')
print(f'  Command:  {" ".join(step1_cmd)}')
print()

result = subprocess.run(
    ' '.join(step1_cmd),
    shell=True, cwd=str(tbss_dir),
    capture_output=True, text=True
)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])
else:
    fa_dir = tbss_dir / 'FA'
    if fa_dir.exists():
        print(f'✓ FA/ directory created with {len(list(fa_dir.glob("*")))} files')
        print('  → Open slicesdir/index.html for visual QC of all FA maps')

## Step 2: tbss_2_reg — register to standard space

Registers every subject's FA to the FMRIB58_FA standard template using FNIRT (nonlinear registration). This is the slowest step (~5-20 min per subject).

In [ ]:
# ─── [FSL] tbss_2_reg ─────────────────────────────────────────────────────────
#
# -t : register to FMRIB58_FA (recommended for human adults)
# -n : register to study-specific template (better for non-adult / clinical)
# -T : already aligned to MNI (skip registration — for HCP preprocessed data)

step2_cmd = ['tbss_2_reg', '-t']

print('[FSL] Step 2: tbss_2_reg -t')
print('  -t  : register to FMRIB58_FA_1mm standard template')
print('  -n  : use for paediatric / clinical / non-standard populations')
print('  -T  : use if data is already in MNI space (e.g. HCP preprocessed)')
print()
print(f'  Run from: {tbss_dir}')
print(f'  Runtime : ~10-30 min per subject (FNIRT nonlinear registration)')
print()
print('>> Uncomment to run:')
# result = subprocess.run(step2_cmd, cwd=str(tbss_dir), capture_output=True, text=True)
# print(result.stdout)

## Step 3 & 4: Create skeleton, project FA

In [ ]:
# ─── [FSL] tbss_3_postreg + tbss_4_prestats ───────────────────────────────────

print('[FSL] Step 3: tbss_3_postreg -S')
print('  Creates: mean_FA.nii.gz — average FA across all subjects')
print('           mean_FA_skeleton.nii.gz — WM skeleton derived from mean FA')
print()
print('[FSL] Step 4: tbss_4_prestats 0.2')
print('  0.2 = FA skeleton threshold (voxels with mean FA < 0.2 excluded)')
print('  Creates: all_FA_skeletonised.nii.gz — one 4D file, all subjects on skeleton')
print()

# Show what the commands look like
for cmd, desc in [
    ('tbss_3_postreg -S',      'Build mean FA and derive skeleton'),
    ('tbss_4_prestats 0.2',    'Project all subjects onto skeleton at FA>0.2'),
]:
    print(f'  $ {cmd}')
    print(f'    → {desc}')
    print()

# For other metrics (MD, RD, AD): after running FA TBSS, project using
print('To also analyse MD/RD/AD (recommended!):')
print('  $ tbss_non_FA MD')
print('  $ tbss_non_FA RD')
print('  $ tbss_non_FA AD')
print('  These project the non-FA metrics using the FA-derived warps')

## Step 5: Statistical testing with randomise

`randomise` is FSL's permutation-based non-parametric statistics tool. It tests for group differences while properly correcting for multiple comparisons using **TFCE** (Threshold-Free Cluster Enhancement).

In [ ]:
# ─── [FSL] randomise ──────────────────────────────────────────────────────────
#
# Required inputs:
#   -i : 4D image (all_FA_skeletonised.nii.gz from step 4)
#   -o : output prefix
#   -m : skeleton mask
#   -d : design matrix (created with Glm_gui or fsl_glm)
#   -t : contrast matrix
#   -n : number of permutations (5000 standard; 500 for quick test)
#   --T2 : use TFCE for multiple comparison correction (recommended)
#   -R : output raw tstat images

print('[FSL] randomise — permutation testing with TFCE')
print()

randomise_cmd = [
    'randomise',
    '-i', 'all_FA_skeletonised.nii.gz',
    '-o', 'tbss_FA',
    '-m', 'mean_FA_skeleton_mask.nii.gz',
    '-d', 'design.mat',    # created with Glm_gui
    '-t', 'design.con',    # contrast file
    '-n', '5000',          # permutations
    '--T2',                # TFCE
    '-R',                  # raw stats
    '-x',                  # output voxel-wise stats
]
print('  Command:', ' '.join(randomise_cmd))
print()
print('  Outputs:')
print('    tbss_FA_tfce_corrp_tstat1.nii.gz  → 1-p value map (TFCE corrected)')
print('    tbss_FA_tstat1.nii.gz             → raw t-statistic map')
print()
print('  To threshold and visualise:')
print('    Open in FSLeyes: overlays tbss_FA_tfce_corrp_tstat1 on mean_FA + skeleton')
print('    Threshold at 0.95 (= p < 0.05 corrected)')
print()

# Design matrix example: 2-group comparison (patients vs controls)
print('=== Design matrix for 2-group comparison ===')
print('(generate with: Glm_gui or Text in FSL)')
print()
n_patients  = 5
n_controls  = 5
design = np.array(
    [[1, 0]] * n_patients +
    [[0, 1]] * n_controls
)
contrast = np.array([[1, -1],   # patients > controls
                     [-1, 1]])  # controls > patients
print('Design matrix (EV1=patients, EV2=controls):')
print(design)
print()
print('Contrast matrix:')
print(contrast)

## Visualise TBSS results

In [ ]:
# Simulate TBSS-style result visualisation
# (In a real study, load actual randomise output)

data_dir_g = Path('../../data/hcp/100307/T1w/Diffusion')
fa_path    = str(Path('../../data/hcp/100307/dti') / 'fsl_dti_FA.nii.gz')

if Path(fa_path).exists():
    fa_img  = nib.load(fa_path)
    fa_data = fa_img.get_fdata()

    # Simulate skeleton (centre voxels of high-FA regions)
    from scipy.ndimage import binary_erosion
    skeleton = (fa_data > 0.3) & ~binary_erosion(fa_data > 0.3, iterations=2)

    # Simulate a "significant" result cluster on skeleton
    sig_mask = skeleton.copy()
    sig_mask[:, :fa_data.shape[1]//2, :] = False   # only right side for demo

    z = fa_data.shape[2] // 2
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, sl_ax, label in zip(axes, [2, 1, 0], ['Axial', 'Coronal', 'Sagittal']):
        idx = fa_data.shape[sl_ax] // 2
        bg  = np.take(fa_data,  idx, axis=sl_ax)
        sk  = np.take(skeleton, idx, axis=sl_ax)
        sig = np.take(sig_mask, idx, axis=sl_ax)

        ax.imshow(bg.T,  cmap='gray',  origin='lower', vmin=0, vmax=0.8)
        ax.imshow(np.ma.masked_where(~sk,  bg * 0 + 0.6).T,
                  cmap='Greens', origin='lower', alpha=0.6, vmin=0, vmax=1)
        ax.imshow(np.ma.masked_where(~sig, sig.astype(float)).T,
                  cmap='hot',   origin='lower', alpha=0.9, vmin=0, vmax=1)
        ax.set_title(f'{label} — FA (grey) + skeleton (green) + significant (hot)')
        ax.axis('off')

    fig.suptitle('TBSS result schematic\n'
                 'Green = WM skeleton | Hot colours = significant group difference',
                 fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('Run Module 2 (DTI) to generate FA maps for visualisation.')

---

## Common TBSS mistakes and how to spot them

| Mistake | What it looks like | How to detect | Fix |
|---|---|---|---|
| Bad registration | Results near ventricles / CSF | Visual check of registered FA | Use `-n` (study-specific template) |
| Wrong FA threshold | Too few / too many skeleton voxels | Check skeleton mask volume | Adjust tbss_4_prestats threshold |
| Missing subjects | N in stats ≠ N expected | Check 4D image shape | Re-run from tbss_1 |
| High motion subjects | Outlier FA values | Check FA histogram per subject | Exclude high-motion subjects |
| Reporting uncorrected | p < 0.05 uncorrected looks great | Check: is it TFCE? | Always use `--T2` / report corrected |

**Next**: [Fixel-Based Analysis →](03_fixel_based_analysis.ipynb)